In [1]:
# Importar paquetes
import os
from pathlib import Path
import import_ipynb
import yaml
import sys
import traceback
import joblib
import mlflow
import mlflow.sklearn
import pandas as pd
from mlflow.models import infer_signature
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from preprocess import load_data, preprocess, split_data

In [2]:
# Definicion de rutas
workspace_dir  = Path.cwd().resolve().parent
mlruns_dir     = os.path.join(workspace_dir, "mlruns")
tracking_uri   = "file:///" + os.path.abspath(mlruns_dir).replace("\\", "/")
artifact_loc   = tracking_uri          # experimentos y modelos en el mismo directorio
model_pkl_path = os.path.join(workspace_dir, "model.pkl")
data_path_dir = os.path.join(workspace_dir,"data")
config_path = os.path.join(workspace_dir, "config.yaml")

print(f"[train] CWD            : {workspace_dir}")
print(f"[train] MLRuns dir     : {mlruns_dir}")
print(f"[train] Tracking URI   : {tracking_uri}")
print(f"[train] Data           : {data_path_dir}")
print(f"[train] Config         : {config_path}")
os.makedirs(mlruns_dir, exist_ok=True)

[train] CWD            : C:\Users\jmanu\Documents\MLops\proyecto_final_v2
[train] MLRuns dir     : C:\Users\jmanu\Documents\MLops\proyecto_final_v2\mlruns
[train] Tracking URI   : file:///C:/Users/jmanu/Documents/MLops/proyecto_final_v2/mlruns
[train] Data           : C:\Users\jmanu\Documents\MLops\proyecto_final_v2\data
[train] Config         : C:\Users\jmanu\Documents\MLops\proyecto_final_v2\config.yaml


In [5]:
# Definhicion del tracking de mlflow
mlflow.set_tracking_uri(tracking_uri)

experiment_name = "CI-CD-Lab-MLflow"
try:
    experiment_id = mlflow.create_experiment(
        name=experiment_name,
        artifact_location=artifact_loc,
    )
    print(f"[train] Experimento creado  → ID: {experiment_id}")
except mlflow.exceptions.MlflowException as exc:
    if "RESOURCE_ALREADY_EXISTS" not in str(exc):
        raise
    exp = mlflow.get_experiment_by_name(experiment_name)
    experiment_id = exp.experiment_id
    print(f"[train] Experimento existente → ID: {experiment_id}")

[train] Experimento creado  → ID: 621127399807356095


C:\Users\jmanu\AppData\Local\Programs\Python\Python311\Lib\site-packages\mlflow\tracking\_tracking_service\utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)


In [7]:
# Parametros globales
with open(config_path, "r") as f:
    config = yaml.safe_load(f)

In [9]:
data_path = os.path.join(data_path_dir,"dataset.csv")

In [11]:
with mlflow.start_run(experiment_id=experiment_id) as run:

    # cargar datos
    df = load_data(data_path) 
    X, y, scaler = preprocess(df, config["data"]["target"])

    X_train, X_test, y_train, y_test = split_data(
        X, y,
        config["model"]["test_size"],
        config["model"]["random_state"]
    )

    # modelo
    model = LinearRegression()
    model.fit(X_train, y_train)

    # predicción
    y_pred = model.predict(X_test)

    # métricas
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)

    # logging MLflow
    mlflow.log_param("model", "LinearRegression")
    mlflow.log_metric("mse", rmse)
    mlflow.log_metric("r2", r2)

    mlflow.sklearn.log_model(
        model,
        "model",
        input_example=X_test[:5]
    )

    print(f"MSE: {mse}")
    print(f"R2: {r2}")

2026/05/03 19:37:49 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/03 19:37:49 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


MSE: 1558239575.1510887
R2: 0.7968483304281098


In [13]:
joblib.dump(model, model_pkl_path)
print(f"[train] model.pkl guardado en: {model_pkl_path}")
print("[train] ✅ Entrenamiento completado exitosamente.")

[train] model.pkl guardado en: C:\Users\jmanu\Documents\MLops\proyecto_final_v2\model.pkl
[train] ✅ Entrenamiento completado exitosamente.
